In [ ]:
!pip install scikit-learn pandas numpy
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Assume data is in /kaggle/working or similar relative path if run locally
DATA_DIR = Path("../../outputs/c3_clean/predictions")
if not DATA_DIR.exists():
    print("Warning: DATA_DIR not found. Paths may need adjustment on Kaggle.")
    
# Import shared utilities
sys.path.append('.')
from utils import compute_all_metrics, to_latex_table

In [ ]:
def load_probs(sys_name, seeds=[42, 1, 7]):
    probs = []
    for s in seeds:
        # Use exact names found in the dir
        fname = f"{sys_name}_seed{s}_test_probs.npy"
        path = DATA_DIR / fname
        if path.exists():
            probs.append(np.load(path))
        else:
            print(f"Missing {path}")
    return probs

y_test = np.load(DATA_DIR / "test_labels.npy")

systems = {
    "A0 (Baseline)": load_probs("A0_Baseline_Clean"),
    "A1 (+ASL)": load_probs("A1_ASL"),
    "A2 (+EmoViS)": load_probs("A2_EmoViS"),
    "A3 (+EmoViS+CB)": load_probs("A3_EmoViS_CB")
}

# Add ensemble
ens_path = DATA_DIR / "ensemble_emoviscb_test_probs.npy"
if ens_path.exists():
    systems["EmoViS-Ens"] = [np.load(ens_path)]


In [ ]:
# Compute metrics
results = []

for sys_name, prob_list in systems.items():
    if not prob_list:
        continue
    
    sys_metrics = {"macro_f1": [], "micro_f1": [], "weighted_f1": [], "map": [], "hamming": [], "em": []}
    
    for probs in prob_list:
        # We assume opt_threshold was tuned on validation. For simplicity in this script 
        # (since we don't have val probs directly here), we use fixed 0.5. 
        # In a real scenario, you'd load val probs to tune. 
        # Let's use 0.5 for all extended metrics except macro_f1 which could use tuning.
        thresholds = np.full(y_test.shape[1], 0.5)
        
        m = compute_all_metrics(probs, y_test, thresholds)
        sys_metrics["macro_f1"].append(m["macro_f1"])
        sys_metrics["micro_f1"].append(m["micro_f1"])
        sys_metrics["weighted_f1"].append(m["weighted_f1"])
        sys_metrics["map"].append(m["map"])
        sys_metrics["hamming"].append(m["hamming_loss"])
        sys_metrics["em"].append(m["exact_match"])
        
    if len(prob_list) > 1:
        # Mean +- std
        row = {"System": sys_name}
        for k in sys_metrics:
            mean = np.mean(sys_metrics[k])
            std = np.std(sys_metrics[k], ddof=1)
            row[k] = f"{mean:.4f} \pm {std:.4f}"
        results.append(row)
    else:
        # Single run (ensemble)
        row = {"System": sys_name}
        for k in sys_metrics:
            row[k] = f"{sys_metrics[k][0]:.4f}"
        results.append(row)

df = pd.DataFrame(results)
df.columns = ["System", "Macro-F1", "Micro-F1", "Weighted-F1", "mAP", "Hamming Loss", "Exact Match"]

# Output latex
print(to_latex_table(df, "Extended Evaluation Metrics on ViGoEmotions Test Set.", "tab:extended_metrics"))
